## Document processing

In [26]:
import json

# ── FIX: always specify encoding="utf-8" on Windows ──────────────
courses_data = json.load(open("shu_rag_chunks.json",      encoding="utf-8"))
supplement   = json.load(open("shu_supplementary_kb.json", encoding="utf-8"))

course_chunks = courses_data["chunks"]          # 6,296 chunks
supp_chunks   = supplement["chunks"]            # 23 chunks
all_chunks    = course_chunks + supp_chunks     # 6,319 total

print(f"Course chunks:      {len(course_chunks)}")
print(f"Supplement chunks:  {len(supp_chunks)}")
print(f"Total:              {len(all_chunks)}")

# Save merged file for ingestion
merged = {
    "total_chunks": len(all_chunks),
    "chunks":       all_chunks
}
with open("shu_all_chunks.json", "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print("Saved → shu_all_chunks.json")

Course chunks:      6296
Supplement chunks:  23
Total:              6319
Saved → shu_all_chunks.json


In [27]:
from langchain_community.document_loaders import JSONLoader
import re

def metadata_func(sample, meta):
    text = sample.get("text", "")
    
    def extract(pattern):
        match = re.search(pattern, text)
        return match.group(1).strip() if match else None

    return {
        **meta,
        **sample.get("metadata", {}),
        "course_name": extract(r"Course:\s*(.*)"),
        "ucas_code": extract(r"UCAS Code:\s*(.*)"),
        "uk_fee": extract(r"UK Fee:\s*£([\d,]+)"),
        "int_fee": extract(r"Int Fee:\s*£([\d,]+)"),
        "location": extract(r"Location:\s*(.*)")
    }

loader = JSONLoader(
    file_path="shu_all_chunks.json",
    jq_schema=".chunks[]",
    content_key="text",
    metadata_func=metadata_func,
    text_content=False
)
docs = loader.load()

In [28]:
print(docs[1])

page_content='Course: Accounting and Finance (BA (Honours) Accounting and Finance)
Subject Area: accounting-banking-and-finance
Module: Business Economics
Year:  | Type: Compulsory modules
Credits: 20 | Assessment: Exam (100%)
Description: The module aims to provide students with a solid foundation in the fundamental concepts and theories of economics. The module introduces students to the principles of micro and macroeconomics. It seeks to equip students with the analytical tools necessary to understand and evaluate economic issues and policies in a wide range of contexts. Additionally, the module aims to cultivate problem-solving skills by encouraging students to apply economic theories to real-world situations.
Topics covered:
  • Demand and Supply Analysis
  • Elasticity of Demand and Supply
  • Consumer Choice Theory
  • Production Theory and Cost Analysis
  • Perfectly Competitive Markets
  • Monopoly and Monopolistic Competition
  • Oligopoly and Game Theory
  • Externalities an

In [29]:
len(docs)

6319

In [30]:
"""
=============================================================================
SHU RAG — Layer 1: Data Ingestion with Metadata
=============================================================================
"""

from langchain_community.document_loaders import JSONLoader
import re


# =============================================================================
# Pattern sets — one per chunk type
# =============================================================================

COURSE_PATTERNS = {
    "degree_type":         r"Degree Type:\s*(.+)",
    "study_mode":          r"Mode:\s*([^|\n]+)",
    "ucas_code":           r"UCAS Code:\s*([A-Z0-9]{4})",
    "entry_requirements":  r"Entry Requirements:\s*(.+)",
    "uk_fee":              r"UK Fee:\s*(£[\d,]+[^|\n]*)",    
    "int_fee":             r"Int Fee:\s*(£[\d,]+[^|\n]*)",
    "placement":           r"Placement:\s*(Yes|No)",
    "location":            r"Location:\s*(.+)",
}


MODULE_PATTERNS = {
              
    "degree_type":     r"Course:.+?\((.+?)\)(?:\s|$)",    
    "module_section":  r"Type:\s*([^|\n]+)\s*modules",    
    "module_credits":  r"Credits:\s*(\d+)",
    "assessment":      r"Assessment:\s*(.+)",
}

# supplementary chunks: free prose — extract only what's reliably present
SUPPLEMENT_PATTERNS = {
    "course_name":  None,   # not applicable
    "ucas_code":    None,
    "uk_fee":       r"£([\d,]+)\s*per year",   # may appear in fees chunks
    "int_fee":      r"international.{0,30}£([\d,]+)",
}

GENERAL_CHUNK_TYPES = {
    "overview", "campus", "admissions", "fees",
    "accommodation", "international", "support",
    "student_life", "contacts",
}
# =============================================================================
# metadata_func — called by JSONLoader for every chunk
# =============================================================================

def metadata_func(sample: dict, meta: dict) -> dict:
    """
    Extracts and promotes all metadata fields from a chunk.

    `sample` = the full chunk dict (one element from chunks[])
    `meta`   = JSONLoader's built-in metadata (source file path, seq_num)

    Returns a flat metadata dict that LangChain attaches to the Document.
    """
    text       = sample.get("text", "")
    chunk_type = sample.get("chunk_type", "")
    meta_sub   = sample.get("metadata", {})   # the nested metadata sub-dict

    # ── Step 1: promote chunk-level fields ────────────────────────────────────
    # These are already structured — no regex needed
    base = {


        # Top-level chunk fields
        "chunk_id":    sample.get("chunk_id", ""),
        "chunk_type":  chunk_type,
        "source_url":  sample.get("source_url", ""),

        # Top-level shorthand fields (present on all chunk types)
        "course":      sample.get("course", ""),
        "subject":     sample.get("subject", ""),
        "module":      sample.get("module", ""),      # only on module_detail
        "year":        sample.get("year", ""),        # module year level

        # Nested metadata sub-dict — already well-structured
        "category":        meta_sub.get("category", ""),
        "course_level":    meta_sub.get("course_level", ""),
        "entry_year":      meta_sub.get("entry_year", ""),
        "confidence":      meta_sub.get("confidence", "high"),

        # Supplementary-specific fields
        "subcategory":     sample.get("subcategory", meta_sub.get("subcategory", "")),
        "target_audience": meta_sub.get("target_audience", "all"),
    }

    # ── Step 2: regex-extract text fields by chunk type ───────────────────────
    def extract(pattern, text=text):
        if pattern is None:
            return ""
        m = re.search(pattern, text, re.IGNORECASE)
        return m.group(1).strip() if m else ""

    if chunk_type == "course_summary":
        base.update({
            "degree_type":        extract(COURSE_PATTERNS["degree_type"]),
            "study_mode":         extract(COURSE_PATTERNS["study_mode"]).strip(" |"),
            "ucas_code":          extract(COURSE_PATTERNS["ucas_code"]),
            "entry_requirements": extract(COURSE_PATTERNS["entry_requirements"]),
            "uk_fee":             extract(COURSE_PATTERNS["uk_fee"]).strip(),
            "int_fee":            extract(COURSE_PATTERNS["int_fee"]).strip(),
            "placement":          extract(COURSE_PATTERNS["placement"]),
            "location":           extract(COURSE_PATTERNS["location"]),
        })

    elif chunk_type == "module_detail":
        base.update({
            "degree_type":     extract(MODULE_PATTERNS["degree_type"]),
            "module_section":  extract(MODULE_PATTERNS["module_section"]),
            "module_credits":  extract(MODULE_PATTERNS["module_credits"]),
            "assessment":      extract(MODULE_PATTERNS["assessment"]),
        })

    elif chunk_type in GENERAL_CHUNK_TYPES:
        base.update({
            "uk_fee":  extract(r"UK Fee:\s*(£[\d,]+[^|\n]*)"),
            "int_fee": extract(r"Int Fee:\s*(£[\d,]+[^|\n]*)"),
        })


    return base


# =============================================================================
# Loader 
# =============================================================================

loader = JSONLoader(
    file_path="shu_all_chunks.json",    
    jq_schema=".chunks[]",              
    content_key="text",                
    metadata_func=metadata_func,
    text_content=False,                 
)

docs = loader.load()

print(f"Loaded {len(docs)} documents")


Loaded 6319 documents


In [31]:
print(docs[-1])

page_content='Sheffield Hallam University key contacts (March 2026):

GENERAL ENQUIRIES:
- Main switchboard: +44 (0)114 225 5555
- General email: enquiries@shu.ac.uk
- Address: City Campus, Howard Street, Sheffield, S1 1WB

STUDENT SERVICES (Hallam Help):
- Phone: 0114 225 4321 (Mon–Fri 9am–5pm)
- Email: hallamhelp@shu.ac.uk
- Location: Owen Building, City Campus (ground floor)
- Online: hallamhelp.shu.ac.uk (MyHallam portal)

ADMISSIONS:
- Undergraduate: 0114 225 5555
- Postgraduate: pg.admissions@shu.ac.uk
- International admissions: international.admissions@shu.ac.uk

INTERNATIONAL STUDENTS:
- International Experience Team: international.experience@shu.ac.uk
- Visa / immigration advice: immigration.adviser@shu.ac.uk

ACCOMMODATION:
- Accommodation office: accommodation@shu.ac.uk
- Online applications: accom-online.shu.ac.uk

FINANCE:
- Student Finance Team: studentfinance@shu.ac.uk
- Scholarship enquiries: funding@shu.ac.uk

WELLBEING & SUPPORT:
- Wellbeing Services: wellbeing@shu.a

In [33]:
# =============================================================================
# Install
# =============================================================================
# pip install sentence-transformers langchain-huggingface

# =============================================================================
# Embeddings — HuggingFace local with batching
# =============================================================================

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # 768-dim, best quality
    model_kwargs={"device": "cpu"},   # swap to "cuda" if GPU available
    encode_kwargs={
        "batch_size": 64,             # 64 docs per batch — good balance for CPU
        "normalize_embeddings": True, # cosine similarity works best normalised
        
    }
)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3282.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:

from langchain_community.vectorstores import Chroma
from tqdm import tqdm

BATCH_SIZE = 500  # Chroma handles up to 5000 but 500 is safer for memory

def ingest_in_batches(docs, embeddings, persist_dir, collection_name, batch_size=500):
    vectorstore = None
    for i in tqdm(range(0, len(docs), batch_size), desc="Ingesting batches"):
        batch = docs[i : i + batch_size]
        # ✅ Stable IDs (important for updates/deduplication)
        ids = [
            doc.metadata.get("chunk_id", f"id_{i+j}")
            for j, doc in enumerate(batch)
        ]
        if vectorstore is None:

            # First batch — create the store
            vectorstore = Chroma.from_documents(
                documents=batch,
                embedding=embeddings,
                ids=ids,
                persist_directory=persist_dir,
                collection_name=collection_name,
            )
        else:
            # Subsequent batches — add to existing store
            vectorstore.add_documents(batch)

    
    return vectorstore

vectorstore = ingest_in_batches(
    docs=docs,
    embeddings=embeddings,
    persist_dir="./shu_vectorstore",
    collection_name="shu_chunks",
)
total_vectors = len(vectorstore.get()["ids"])

print(f"Ingestion complete — {total_vectors} vectors stored")

Ingesting batches: 100%|██████████| 13/13 [02:50<00:00, 13.10s/it]


Ingestion complete — 6319 vectors stored


In [35]:
# Sanity check — should match len(docs)
if total_vectors != len(docs):
    print(f"WARNING: expected {len(docs)}, got {total_vectors} — possible ID collision or missing chunks")
else:
    print("All chunks ingested successfully")


All chunks ingested successfully


In [102]:

from pydantic import BaseModel, Field, field_validator
from typing import List, Optional, Dict, Any
from langchain_classic.schema import Document


# =============================================================================
# Pydantic output schema
# =============================================================================

class QueryIntent(BaseModel):
    intents: List[str] = Field(
        description=(
            "One or more retrieval intents. Each must be one of: "
            "'course_summary', 'module_detail', 'general'. "
            "Include multiple if the query spans more than one area."
        )
    )
    k: int = Field(
        description=(
            "Number of documents to retrieve based on query complexity:"
        )
    )
    filters: Dict[str, Any] = Field(
        default_factory=dict,
        description=(
            "Metadata filters to apply during vector store retrieval. "
            "Only include fields explicitly mentioned or clearly implied by the query. "
            "Empty dict if no filters apply. "
            "\n\nAvailable filter keys and their valid values:"
            "\n  Shared:"
            "\n    'course'         : exact course name in title case e.g. 'Computer Science'"
            "\n    'subject'        : subject slug e.g. 'computing', 'accounting-banking-and-finance'"
            "\n    'course_level'   : 'undergraduate' or 'postgraduate'"
            "\n  course_summary only:"
            "\n    'entry_year'     : always '2026' for course_summary queries"
            "\n    'degree_type'    : e.g. 'BA (Honours)', 'BSc (Honours)', 'MSc'"
            "\n    'study_mode'     : 'full-time' or 'part-time'"
            "\n    'placement'      : 'Yes' or 'No'"
            "\n    'location'       : 'City Campus' or 'Collegiate Campus'"
            "\n  module_detail only:"
            "\n    'module'         : exact module name in title case e.g. 'Business Economics'"
            "\n    'year'           : module year level '1', '2', or '3'"
            "\n    'module_section' : 'Compulsory' or 'Optional'"
            "\n    'assessment'     : e.g. 'Exam (100%)', 'Coursework (100%)'"
            "\n  general only:"
            "\n    'subcategory'    : e.g. 'key_contacts', 'accommodation'"
            "\n    'target_audience': 'international', 'domestic', or 'all'"
        )
    )
    rewritten_query: str = Field(
        description=(
            "Clean, standalone, expanded query for vector search. "
            "Expand abbreviations, fix typos, resolve pronouns, add SHU context. "
            "e.g. 'CS fees?' → 'What are the tuition fees for the Computer Science "
            "degree at Sheffield Hallam University?'"
        )
    )
    is_greeting_or_chitchat: bool = Field(
        default=False,
        description=(
            "True if the query is a greeting, chitchat, or completely off-topic. "
            "No retrieval needed."
        )
    )

    @field_validator("intents")
    @classmethod
    def validate_intents(cls, v, values):
        # If it's chitchat → allow empty intents
        if values.data.get("is_greeting_or_chitchat"):
            return v or ["general"]  # safe fallback

        valid = {"course_summary", "module_detail", "general"}
        invalid = [i for i in v if i not in valid]
        
        if invalid:
            raise ValueError(f"Invalid intents: {invalid}")
        if not v:
            raise ValueError("intents must not be empty")
        
        return v

    @field_validator("k")
    @classmethod
    def validate_k(cls, v):
        return max(3, min(40, v))

In [103]:
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import SystemMessage, HumanMessage

# =============================================================================
# LLM — ChatBedrockConverse
# =============================================================================

llm = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1",
)

# =============================================================================
# Structured LLM
# =============================================================================

structured_llm = llm.with_structured_output(QueryIntent)


In [104]:
# =============================================================================
# System prompt
# =============================================================================

# =============================================================================
# System prompt — full metadata awareness
# =============================================================================

SYSTEM_PROMPT = """You are a query processor for Sheffield Hallam University's AI assistant.

Your job is to analyse the user's question and populate ALL structured output fields.
The output drives metadata filtering and vector search against the university's scraped content.

=== AVAILABLE DATA ===
The knowledge base contains chunks from shu.ac.uk with these chunk types:

  course_summary  — fees (UK and international), UCAS codes, entry requirements,
                    placement year, campus location, degree type, study mode, duration

  module_detail   — module name, year level, topics covered, credits,
                    assessment method, compulsory or optional

  general         — contacts, accommodation, admissions, visa, international students,
                    wellbeing, support, student life, scholarships and funding

=== INTENT ===
Identify which chunk type(s) are needed. Use multiple intents if the query spans areas.

  course_summary → fees, tuition, entry requirements, UCAS code, placement,
                   campus location, degree type, study mode, duration
  module_detail  → modules, topics, credits, assessment, syllabus,
                   compulsory or optional, what is taught, year level
  general        → contacts, email, accommodation, visa, international,
                   support, wellbeing, careers, funding
IMPORTANT RULE
If the query is a greeting, identity question, or unrelated to university content:
  - Set is_greeting_or_chitchat = true
  - Set intents = ["general"]
  - Set k = 3
  - filters = {}
                   
=== K VALUE GUIDELINES ===

Select k based on BOTH query complexity AND intent type.

k guidance — set based on query type:

  3  → single fact lookup
       e.g. "What is the UCAS code for Nursing?"
       e.g. "Is there a placement year on Computer Science?"

  5  → single aspect, specific course
       e.g. "What are the entry requirements for Computer Science?"
       e.g. "What is the UK tuition fee for Nursing?"

  15 → broad overview of a single course
       e.g. "Tell me about the Computer Science degree"
       e.g. "What is the Accounting and Finance course like?"

  25 → list-type queries — retrieving ALL items of a category
       e.g. "What modules are available in Computer Science?"
       e.g. "What are all the compulsory modules in Nursing?"
       e.g. "List all the optional modules in year 2"

  40 → multi-course or multi-aspect list queries
       e.g. "Compare the modules in Computer Science and Software Engineering"
       e.g. "What modules are available across all computing courses?"

IMPORTANT:
  - For ANY query asking for a list of modules, topics, or courses — always use k >= 25
  - Higher k = better recall — the reranker will filter down to the most relevant
  - It is always better to retrieve too many than too few
  - The reranker runs AFTER retrieval and selects the best documents from the pool
  - Never use k=8 for list-type queries — you will miss relevant documents


=== FILTERS DICT ===
Populate the filters dict with ONLY the keys that are explicitly mentioned
or clearly implied by the query. Leave the dict empty if nothing specific is mentioned.

  Shared keys (apply to any intent):
    "course"          exact course name in title case      e.g. "Computer Science"
    "subject"         subject slug lowercase-hyphenated    e.g. "accounting-banking-and-finance"
    "course_level"    level of study                       "undergraduate" or "postgraduate"

  course_summary keys:
    "entry_year"      always include for course queries    always "2026"
    "degree_type"     degree type if mentioned             e.g. "BSc (Honours)"
    "study_mode"      mode of study if mentioned           "full-time" or "part-time"
    "placement"       placement year if mentioned          "Yes" or "No"
    "location"        campus if mentioned                  "City Campus" or "Collegiate Campus"

  module_detail keys:
    "module"          exact module name in title case      e.g. "Business Economics"
    "year"            module year level if mentioned       "1", "2", or "3"
    "module_section"  module type if mentioned             "Compulsory" or "Optional"
    "assessment"      assessment type if mentioned         e.g. "Exam (100%)"

  general keys:
    "subcategory"     specific info area if clear          e.g. "key_contacts", "accommodation"
    "target_audience" audience if mentioned                "international", "domestic", or "all"

Examples:
  Query: "What are the full-time undergraduate fees for Computer Science?"
  filters: {"course": "Computer Science", "course_level": "undergraduate",
             "study_mode": "full-time", "entry_year": "2026"}

  Query: "What compulsory modules are in year 2 of Nursing?"
  filters: {"course": "Nursing", "year": "2", "module_section": "Compulsory"}

  Query: "How do I apply for accommodation?"
  filters: {"subcategory": "accommodation"}

  Query: "Tell me about Sheffield Hallam University"
  filters: {}

=== QUERY REWRITING ===
Rewrite into a clean, standalone, semantically rich sentence for vector search.
  - Expand abbreviations: "CS" → "Computer Science", "SHU" → "Sheffield Hallam University"
  - Fix typos and grammar
  - Resolve pronouns using context
  - Add university context when a course or module is mentioned
  - Write a single fluent sentence — no bullet points

=== GREETING / CHITCHAT ===
Set is_greeting_or_chitchat to true for greetings, small talk, or completely off-topic queries.
"""


In [106]:

# =============================================================================
# Process query
# =============================================================================

def process_query(query: str) -> QueryIntent:
    try:
        return structured_llm.invoke([
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=query),
        ])
    except Exception as e:
        print(f"[QueryProcessor] fallback triggered: {e}")
    return QueryIntent(
        intents=["general"],
        k=12,
        filters={},
        rewritten_query=query,
        is_greeting_or_chitchat=False,
    )


# =============================================================================
# Test
# =============================================================================

result = process_query("What are the modules present in Computer Science?")
print(result)

intents=['module_detail'] k=25 filters={'course': 'Computer Science'} rewritten_query='What are the modules present in the Computer Science course at Sheffield Hallam University?' is_greeting_or_chitchat=False


In [107]:
def map_intents_to_filters(qp: QueryIntent):
    filters = qp.filters.copy()

    if len(qp.intents) == 1:
        filters["chunk_type"] = qp.intents[0]

    return filters

In [108]:
map_intents_to_filters(result)

{'course': 'Computer Science', 'chunk_type': 'module_detail'}

In [109]:


GENERAL_CHUNK_TYPES = [
    "overview", "campus", "admissions", "fees",
    "accommodation", "international", "support",
    "student_life", "contacts",
]

CHUNK_TYPE_MAP = {
    "course_summary": ["course_summary"],
    "module_detail":  ["module_detail"],
    "general":        GENERAL_CHUNK_TYPES,
}

ALLOWED_FILTER_KEYS = {
    "course", "subject", "course_level", "entry_year",
    "degree_type", "study_mode", "placement", "location",
    "module", "year", "module_section", "assessment",
    "subcategory", "target_audience",
}

In [110]:
def build_chroma_filter(intents: List[str], filters: dict) -> dict:
    # Map intents to chunk_types
    chunk_types = []
    for intent in intents:
        if intent in CHUNK_TYPE_MAP:
            chunk_types.extend(CHUNK_TYPE_MAP[intent])

    # chunk_type filter
    if len(chunk_types) == 1:
        conditions = [{"chunk_type": {"$eq": chunk_types[0]}}]
    else:
        conditions = [{"chunk_type": {"$in": chunk_types}}]

    # entry_year default for course_summary
    if "course_summary" in intents and "entry_year" not in filters:
        conditions.append({"entry_year": {"$eq": "2026"}})

    # wrap each filter value in $eq
    for key, value in filters.items():
        if value is not None:
            conditions.append({key: {"$eq": value}})

    # for key, value in filters.items():
    #     if key in ALLOWED_FILTER_KEYS and value is not None:
    #         conditions.append({key: {"$eq": value}})
    
    return {"$and": conditions} if len(conditions) > 1 else conditions[0]

In [ ]:
# def get_retriever(vectorstore, query_intent: QueryIntent):
#     intents = query_intent.intents
#     k = query_intent.k
#     filters = query_intent.filters or {}

#     where = build_chroma_filter(intents, filters)

#     if "module_detail" in intents:
#         return vectorstore.as_retriever(
#             search_type="mmr",
#             search_kwargs={
#                 "k": k,
#                 "fetch_k": k * 2,
#                 "filter": where,
#             },
#         )
#     else:
#         return vectorstore.as_retriever(
#             search_type="similarity",
#             search_kwargs={
#                 "k": k,
#                 "filter": where,
#             },
#         )

In [111]:
def get_retriever_with_fallback(vectorstore, query_intent: QueryIntent):
    intents     = query_intent.intents
    k           = query_intent.k
    filters     = query_intent.filters or {}
    search_type = "mmr" if "module_detail" in intents else "similarity"

    where = build_chroma_filter(intents, filters)

    # Primary — filtered retrieval
    retriever = vectorstore.as_retriever(
        search_type=search_type,
        search_kwargs={"k": k, "filter": where},
    )

    docs = retriever.invoke(query_intent.rewritten_query)

    # Fallback — filter failed, retry with chunk_type only
    if not docs:
        print("[Retriever] filtered search returned nothing — falling back to chunk_type only")

        chunk_types = []
        for intent in intents:
            if intent in CHUNK_TYPE_MAP:
                chunk_types.extend(CHUNK_TYPE_MAP[intent])

        fallback_filter = (
            {"chunk_type": chunk_types[0]}
            if len(chunk_types) == 1
            else {"chunk_type": {"$in": chunk_types}}
        )

        retriever = vectorstore.as_retriever(
            search_type=search_type,
            search_kwargs={"k": k, "filter": fallback_filter},
        )
        docs = retriever.invoke(query_intent.rewritten_query)

    # Last resort — no filter at all, pure vector search
    if not docs:
        print("[Retriever] chunk_type fallback also failed — falling back to pure vector search")
        retriever = vectorstore.as_retriever(
            search_type=search_type,
            search_kwargs={"k": k},
        )
        docs = retriever.invoke(query_intent.rewritten_query)

    return docs

In [131]:
query_intent = process_query("What are the modules available in ba honours accounting accounting and finance?")
docs         = get_retriever_with_fallback(vectorstore, query_intent)

[Retriever] filtered search returned nothing — falling back to chunk_type only


In [132]:
docs

[Document(metadata={'module_credits': '40', 'target_audience': 'all', 'module_section': 'Compulsory', 'category': 'modules', 'chunk_id': 'c00992', 'module': 'Accounting In Context', 'course_level': 'undergraduate', 'source_url': 'https://www.shu.ac.uk/courses/business-and-management/ba-honours-business-management-and-finance-with-foundation-year/full-time/2026', 'subcategory': 'Business Management and Finance with Foundation Year — Accounting In Context', 'entry_year': '', 'assessment': 'Coursework (70%)', 'year': '', 'chunk_type': 'module_detail', 'subject': 'business-and-management', 'confidence': 'high', 'degree_type': 'BA (Honours)', 'course': 'Business Management and Finance with Foundation Year'}, page_content='Course: Business Management and Finance with Foundation Year (BA (Honours))\nSubject Area: business-and-management\nModule: Accounting In Context\nYear:  | Type: Compulsory modules\nCredits: 40 | Assessment: Coursework (70%)\nDescription: This module develops your knowledg

In [133]:
query_intent

QueryIntent(intents=['module_detail'], k=25, filters={'course': 'BA (Honours) Accounting and Finance'}, rewritten_query='What are the modules available in BA (Honours) Accounting and Finance?', is_greeting_or_chitchat=False)

In [187]:
from sentence_transformers import CrossEncoder

# =============================================================================
# Load cross-encoder reranker
# =============================================================================

reranker = CrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
)


# =============================================================================
# Rerank function
# =============================================================================

def rerank(query: str, docs: List[Document], top_n: int = None) -> List[Document]:
    if not docs:
        return []

    # Scale top_n with corpus size if not specified
    if top_n is None:
        if len(docs) <= 5:
            top_n = len(docs)
        elif len(docs) <= 15:
            top_n = 8
        else:
            top_n = 12   # for large module list queries, keep top 10

    # if top_n is None:
    #     top_n = len(docs)

    pairs  = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    for doc, score in zip(docs, scores):
        doc.metadata["_rerank_score"] = round(float(score), 4)

    reranked = sorted(docs, key=lambda d: d.metadata["_rerank_score"], reverse=True)

    return reranked[:top_n]

The CrossEncoder `model_name` argument was renamed and is now deprecated, please use `model_name_or_path` instead.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3911.38it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
query_intent = process_query("Does the MSc Artificial Intelligence course offer a placement year ?")
docs         = get_retriever_with_fallback(vectorstore, query_intent)
reranked_docs = rerank(query_intent.rewritten_query, docs)

In [223]:
docs

[Document(metadata={'subcategory': 'Data Science and Artificial Intelligence — Foundations Of Data Science For Machine Learning', 'category': 'modules', 'target_audience': 'all', 'subject': 'computing', 'module_credits': '40', 'source_url': 'https://www.shu.ac.uk/courses/computing/msc-data-science-and-artificial-intelligence/full-time/2026', 'chunk_type': 'module_detail', 'degree_type': 'MSc Data Science and Artificial Intelligence', 'course': 'Data Science and Artificial Intelligence', 'module': 'Foundations Of Data Science For Machine Learning', 'entry_year': '', 'module_section': 'Compulsory', 'course_level': 'postgraduate_taught', 'confidence': 'high', 'chunk_id': 'c01941', 'assessment': 'Practical (100%)', 'year': '', '_rerank_score': 1.3913}, page_content='Course: Data Science and Artificial Intelligence (MSc Data Science and Artificial Intelligence)\nSubject Area: computing\nModule: Foundations Of Data Science For Machine Learning\nYear:  | Type: Compulsory modules\nCredits: 40 

In [224]:
reranked_docs

[Document(metadata={'chunk_type': 'module_detail', 'source_url': 'https://www.shu.ac.uk/courses/computing/bsc-honours-data-science/full-time/2026', 'year': '', 'assessment': 'Coursework (100%)', 'entry_year': '', 'target_audience': 'all', 'module_credits': '40', 'module_section': 'Compulsory', 'course': 'Data Science', 'course_level': 'undergraduate', 'chunk_id': 'c01775', 'subcategory': 'Data Science — Programming For Artificial Intelligence', 'module': 'Programming For Artificial Intelligence', 'subject': 'computing', 'degree_type': 'BSc (Honours', 'confidence': 'high', 'category': 'modules', '_rerank_score': 1.7627}, page_content='Course: Data Science (BSc (Honours) Data Science)\nSubject Area: computing\nModule: Programming For Artificial Intelligence\nYear:  | Type: Compulsory modules\nCredits: 40 | Assessment: Coursework (100%)\nDescription: This module enables students to explore cutting-edge programming techniques for artificial intelligence. Students will engage with complex r

In [225]:
query_intent

QueryIntent(intents=['module_detail'], k=40, filters={}, rewritten_query='Compare the modules available in the Data Analytics and AI courses at Sheffield Hallam University.', is_greeting_or_chitchat=False)

In [226]:
ANSWER_SYSTEM_PROMPT = """You are an AI assistant for Sheffield Hallam University.

You MUST answer the question using ONLY the provided context.

Instructions:
- Carefully read all the context chunks
- Extract the relevant information
- If multiple chunks contain useful information, combine them
- Be precise and complete
- Always mention the source URL at the end of your answer
- Never make up fees, entry requirements, UCAS codes, or module names

IMPORTANT:
- Do NOT say "I don't know" if the answer is present in the context
- Only say "I don't know" if the information is truly missing
"""

In [227]:
messages = [
    SystemMessage(content=ANSWER_SYSTEM_PROMPT),
    HumanMessage(content=(
        f"Context:\n{reranked_docs}\n\n"
        f"Question: {query_intent.rewritten_query}"
    )),
]

response = llm.invoke(messages)

In [228]:
print(response.content)

Sheffield Hallam University offers a range of Data Analytics and AI-focused courses, each with distinct modules tailored to different levels and specializations:

**1. Data Science (BSc (Honours) Data Science):**
   - **Module: Programming For Artificial Intelligence**
     - Credits: 40
     - Assessment: Coursework (100%)
     - Topics Covered: Advanced programming techniques, machine learning, deep learning, big data processing, data visualization, API development, cloud AI platforms, model deployment, ethics, and professional development.

**2. Data Science with Foundation Year (BSc (Honours) Data Science with Foundation Year):**
   - **Module: Programming For Artificial Intelligence**
     - Credits: 40
     - Assessment: Coursework (100%)
     - Topics Covered: Same as the standard Data Science course.

**3. Business Management and Artificial Intelligence (BSc (Honours)):**
   - **Module: Ai Data Driven Insights For Business**
     - Credits: 20
     - Assessment: Practical (100%

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

# =============================================================================
# LLM — Nova Pro for answer generation
# =============================================================================

generator_llm = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0",   #amazon.nova-lite-v1:0
    region_name="us-east-1",
    temperature=0.3,
)

# =============================================================================
# System prompt
# =============================================================================

ANSWER_SYSTEM_PROMPT = """You are a helpful AI assistant for Sheffield Hallam University (SHU).
Your job is to answer prospective and current students' questions accurately using 
the provided context retrieved from the official SHU website.

RULES:
  - Answer ONLY from the provided context — do not use outside knowledge
  - If the context does not contain enough information to answer, say:
    "I don't have enough information to answer that. Please visit shu.ac.uk or 
     contact the admissions team directly."
  - Be concise and direct — students want quick, clear answers
  - For list-type answers (modules, topics), present them as a clean list
  - Always mention the source URL at the end of your answer
  - Never make up fees, entry requirements, UCAS codes, or module names
  - If multiple courses are in the context, clearly distinguish between them
"""

# =============================================================================
# Format retrieved docs into context string
# =============================================================================

def format_context(docs: List[Document]) -> str:
    context_parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source_url", "shu.ac.uk")
        context_parts.append(
            f"[Document {i}]\n{doc.page_content}\nSource: {source}"
        )
    return "\n\n---\n\n".join(context_parts)


# =============================================================================
# Answer generation
# =============================================================================

def generate_answer(query_intent: QueryIntent, docs: List[Document]) -> str:
    if not docs:
        return (
            "I couldn't find relevant information for your query. "
            "Please visit shu.ac.uk or contact the admissions team at "
            "enquiries@shu.ac.uk or call +44 (0)114 225 5555."
        )

    context = format_context(docs)

    messages = [
        SystemMessage(content=ANSWER_SYSTEM_PROMPT),
        HumanMessage(content=(
            f"Context:\n{context}\n\n"
            f"Question: {query_intent.rewritten_query}"
        )),
    ]

    response = generator_llm.invoke(messages)
    return StrOutputParser().invoke(response)


# =============================================================================
# Full pipeline
# =============================================================================

def ask(query: str) -> str:
    # Step 1 — classify and enrich query
    query_intent = process_query(query)
    print(query_intent)
    # Step 2 — handle greetings directly
    if query_intent.is_greeting_or_chitchat:
        return "Hello! I'm the Sheffield Hallam University assistant. How can I help you today?"

    # Step 3 — retrieve
    docs = get_retriever_with_fallback(vectorstore, query_intent)

    print(docs)
    # Step 4 — rerank
    reranked_docs = rerank(query_intent.rewritten_query, docs)

    print(reranked_docs)
    
    # Step 5 — generate
    answer = generate_answer(query_intent, reranked_docs)

    return answer


# =============================================================================
# Test
# =============================================================================

print(ask("What are the tuition fees for Computer Science?"))


intents=['course_summary'] k=5 filters={'course': 'Computer Science', 'entry_year': '2026'} rewritten_query='What are the tuition fees for the Computer Science degree at Sheffield Hallam University for the entry year 2026?' is_greeting_or_chitchat=False
[Document(metadata={'year': '', 'uk_fee': '£9,790 per year', 'chunk_id': 'c01597', 'degree_type': 'BSc (Honours) Computer Science', 'subject': 'computing', 'placement': 'Yes', 'course': 'Computer Science', 'subcategory': '', 'entry_year': '2026', 'module': '', 'ucas_code': 'G400', 'int_fee': '£18,000 per year', 'confidence': 'high', 'chunk_type': 'course_summary', 'category': 'course_info', 'source_url': 'https://www.shu.ac.uk/courses/computing/bsc-honours-computer-science/full-time/2026', 'entry_requirements': '112-120 UCAS points', 'study_mode': 'full-time', 'location': 'City Campus', 'course_level': 'undergraduate', 'target_audience': 'all'}, page_content='Course: Computer Science\nSubject Area: computing\nDegree Type: BSc (Honours) 

In [145]:
print(ask("What are the modules available in accounting accounting and finance?"))


intents=['module_detail'] k=25 filters={'course': 'Accounting and Finance'} rewritten_query='What modules are available in the Accounting and Finance course at Sheffield Hallam University?' is_greeting_or_chitchat=False
[Document(metadata={'module_credits': '40', 'chunk_id': 'c00007', 'subcategory': 'Accounting and Finance — Management Accounting Applications', 'target_audience': 'all', 'module': 'Management Accounting Applications', 'course': 'Accounting and Finance', 'confidence': 'high', 'entry_year': '', 'chunk_type': 'module_detail', 'course_level': 'undergraduate', 'module_section': 'Compulsory', 'assessment': 'Exam (60%)', 'subject': 'accounting-banking-and-finance', 'degree_type': 'BA (Honours', 'year': '', 'category': 'modules', 'source_url': 'https://www.shu.ac.uk/courses/accounting-banking-and-finance/ba-honours-accounting-and-finance/full-time/2026'}, page_content='Course: Accounting and Finance (BA (Honours) Accounting and Finance)\nSubject Area: accounting-banking-and-fin

In [130]:
print(ask("How do I apply for accommodation?"))

[Retriever] filtered search returned nothing — falling back to chunk_type only
To apply for accommodation at Sheffield Hallam University, follow these steps:

1. Apply online at: [accom-online.shu.ac.uk](https://accom-online.shu.ac.uk)
2. Apply as soon as you receive an offer from SHU.
3. You do not need to sign contracts or pay a deposit until your results are confirmed.
4. List your preferred properties in order; allocation is based on preferences and availability.

Source: [https://www.shu.ac.uk/study-here/accommodation](https://www.shu.ac.uk/study-here/accommodation)
